In [1]:
import os,sys,json,base64,zipfile,io,hashlib,time
from pathlib import Path
print('START SPI v3 immutable source setup',flush=True)
ROOT=Path('/kaggle/working/spi_source_run20261028_spi_grounding_v1')
ROOT.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as archive:
    if any(not (ROOT/name).resolve().is_relative_to(ROOT) for name in archive.namelist()):raise ValueError('unsafe source archive')
    archive.extractall(ROOT)
SOURCE=json.loads((ROOT/'SOURCE_MANIFEST.json').read_text())
for name,expected in SOURCE['files'].items():
    if hashlib.sha256((ROOT/name).read_bytes()).hexdigest()!=expected:raise ValueError('source checksum mismatch')
sys.path.insert(0,str(ROOT))
os.environ['BETTER_SOURCE_MANIFEST']=str(ROOT/'SOURCE_MANIFEST.json')
os.environ['HF_HOME']='/kaggle/temp/spi_models'
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'):os.environ[key]='1'
from parameters.cloud_notebook import CellRecorder
recorder=CellRecorder(CELL_SOURCES,Path('/kaggle/working/experiments/spi_four_class/runs/run20261028_spi_grounding_v1/cell_evidence'))
with recorder.cell(0):
    print('Source',SOURCE['commit'],'files',len(SOURCE['files']),flush=True)
    print('Interpreter',sys.executable,sys.version,flush=True)
    print('Thinking OFF; one GPU1 quantized teacher at a time; cached CLIP; no YOLO or scheduler training',flush=True)


CELL START 0 log /kaggle/working/experiments/spi_four_class/runs/run20261028_spi_grounding_v1/cell_evidence/cell_0.log
Source 032c23a3dafc7bfeb39a84ba55193b6b6f3a22a0 files 110
Interpreter /usr/bin/python3 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Thinking OFF; one GPU1 quantized teacher at a time; cached CLIP; no YOLO or scheduler training
CELL END 0 seconds 0.010822019999977783


In [2]:
with recorder.cell(1):
    import subprocess,importlib.metadata as metadata,importlib.util
    import torch
    print('START dependency/hardware preflight; retain managed Torch/CUDA',flush=True)
    if torch.cuda.device_count()!=2:raise RuntimeError('existing notebook contract requires two GPUs')
    print('GPUs',[torch.cuda.get_device_name(i) for i in (0,1)],flush=True)
    result=subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',
        'transformers==5.3.0','huggingface-hub==1.5.0','accelerate==1.12.0',
        'bitsandbytes==0.49.2','sentencepiece==0.2.2','scikit-learn==1.7.2',
        'nbformat==5.10.4'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=360)
    print(result.stdout,flush=True);result.check_returncode()
    from transformers import Qwen3_5ForConditionalGeneration,Qwen3VLForConditionalGeneration
    if any(importlib.util.find_spec(x) is not None for x in ('fla','causal_conv1d')):
        raise RuntimeError('unverified optional DeltaNet kernels inherited')
    from parameters.artifacts import write_json
    write_json(Path('/kaggle/working/runtime.json'),{'python':sys.version,'interpreter':sys.executable,
        'packages':{p:metadata.version(p) for p in ('torch','numpy','transformers','huggingface-hub','accelerate','bitsandbytes','scikit-learn','nbformat')},
        'gpus':[torch.cuda.get_device_name(i) for i in (0,1)],'cuda':torch.version.cuda,'thinking':False})
    import shutil
    shutil.copy2('/kaggle/working/runtime.json',recorder.output.parent/'runtime.json')
    print('END dependency/hardware preflight',flush=True)


CELL START 1 log /kaggle/working/experiments/spi_four_class/runs/run20261028_spi_grounding_v1/cell_evidence/cell_1.log
START dependency/hardware preflight; retain managed Torch/CUDA
GPUs ['Tesla T4', 'Tesla T4']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 91.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 20.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 98.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 86.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.6 MB/s  0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: nbformat
    Found existing installation: nbformat 5.11.1
    Uninstalling nbformat-5.11.1:
      Successfully uninstalled nbformat-5.11.1
  Attempting uninstall: huggingface-hub
    Found existing installation

In [3]:
with recorder.cell(2):
    import signal
    from parameters.cloud_notebook import locate
    from experiments.spi_four_class.visual_reliability import execute
    def cutoff(signum,frame):raise TimeoutError('whole-cell budget exceeded')
    signal.signal(signal.SIGALRM,cutoff);signal.alarm(3120)
    INPUT_ROOT=locate('input_manifest.json').parent
    OUTPUT=execute(INPUT_ROOT,Path('/kaggle/working/experiments/spi_four_class/runs/run20261028_spi_grounding_v1/campaign'),
        seconds=3060,maximum_images=300,runtime_receipt=recorder.output.parent/'runtime.json')
    signal.alarm(0)
    print('ACTUAL SPI OUTCOME',json.loads((OUTPUT/'status.json').read_text()),flush=True)


CELL START 2 log /kaggle/working/experiments/spi_four_class/runs/run20261028_spi_grounding_v1/cell_evidence/cell_2.log
START 01_verify_frozen_diagnostic output /kaggle/working/experiments/spi_four_class/runs/run20261028_spi_grounding_v1/campaign
END 01_verify_frozen_diagnostic {'stage': '01_verify_frozen_diagnostic', 'status': 'complete', 'seconds': 62.76806042200002}
START 02_load_qwen35_9b output /kaggle/working/experiments/spi_four_class/runs/run20261028_spi_grounding_v1/campaign
Loading teacher qwen35_9b revision c202236235762e1c871ad0ccb60c8ee5ba337b9a
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
END 02_load_qwen35_9b {'stage': '02_load_qwen35_9b', 'status': 'complete', 'seconds': 207.68168441300003}
START 03_text_conformance_qwen35_9b output /kaggle/working/experiments/spi_fou

ValueError: supply images for visual review, or explicitly request the text-only diagnostic

In [ ]:
with recorder.cell(3):
    print('Prepare recovery export after cell completion; preserve final cell status/logs',flush=True)
# Package after CellRecorder closes so the bundle contains the completed cell,
# rather than a stale running status and a log ending at CELL START.
import zipfile
run=Path('/kaggle/working/experiments/spi_four_class/runs/run20261028_spi_grounding_v1')
files=[p for p in run.rglob('*') if p.is_file() and '/data/images/' not in str(p)
       and p.name not in ('run_recovery.zip','retrieval_checksums.json')]
from parameters.artifacts import write_json
write_json(run/'retrieval_checksums.json',{str(p.relative_to(run)):hashlib.sha256(p.read_bytes()).hexdigest() for p in files})
with zipfile.ZipFile('/kaggle/working/run_recovery.zip','w',zipfile.ZIP_DEFLATED) as archive:
    for p in files:archive.write(p,str(p.relative_to(run)))
    archive.write(run/'retrieval_checksums.json','retrieval_checksums.json')
print('END recovery export; inspect campaign/status.json and verify checksums on retrieval',flush=True)
